In [1]:
print('Product query agent')

Product query agent


In [38]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [72]:
PRODUCTs = {
    "wireless headphones":{"price":79.99, "description":"Over-ear Bluetooth , 30-Hrs battery , Active noise cancellation."},
    "smart watch": {"price":199.99, "description": "Tracks heart rate and sleep . 5-days battery , water-resistant" },
    "mechanical keyboard": {"price":129.00, "description": "Tenkeyless , Cherry MX Brown switches , per-keey RGB"},
    "laptop stand": {"price":34.99, "description": "Adjustable aluminium , fits 11-17 inches laptops , folds flat"},
}



@tool
def get_product(name : str) -> str:
    """ Look up a product by name and return its price , rating , stock and description."""
    p = PRODUCTs.get(name.lower())
    if not p:
        return f"product not Available : {', '.join(PRODUCTs)}"
    return str(p)



# llm = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash',temperature=0)
llm = ChatGroq(model='llama-3.3-70b-versatile',temperature=0)

agent = create_agent(
    llm,
    tools=[get_product],
    system_prompt='you are a helpful product assestant for an online tech store. answer in few lines structured format answer',
)

In [73]:
def ask(question:str):
    result = agent.invoke({'messages':[{"role":"user","content":question}]})
    print(result['messages'][-1].content)

In [74]:
ask("what is the price of Wireless Headphones")

The price of Wireless Headphones is $79.99.


In [75]:
ask("what is the price of smart watch")

The price of the smart watch is $199.99.


In [ ]:
#ask("what mechanical keyboard can do")

A mechanical keyboard can offer customizable backlighting, macro keys, and switch types (like Cherry MX Brown) for improved typing experience and productivity.


In [ ]:
#ask("How a laptop stand be used")

A laptop stand can be used to:
* Elevate the laptop to a comfortable viewing height
* Improve airflow and reduce overheating
* Free up desk space
* Enhance typing experience with an external keyboard
* Create a more ergonomic workspace.


In [ ]:
#ask('tell me about humanbeings')

Humans are complex beings with physical, emotional, and social needs. They have a unique capacity for thought, language, and culture. If you're looking for more information on a specific product related to human well-being, I can try to help you with that.


In [79]:
# execution steps 

# 1. extract producct names from questions:
# 2. get_product("wireless headphones")
# 3. analyze the answer: {"price":79.99, "description":"Over-ear Bluetooth , 30-Hrs battery , Active noise cancellation"}
# 4. return human readable answers

In [90]:
## Reviews



reviews = { 
    "wireless headphones": {"reviews":1234,"rating":4.6},
    "smart watch": {"reviews":555, "rating":4.4},
    "mechanical keyboard":{"reviews":44,"rating":3.2},
    "laptop stand":{"reviews":2343,"rating":4.2}
}

@tool
def get_review(name:str):
    """look up a product review by a product name , and return the product name , number of reviews and rating"""
    r = reviews.get(name.lower())
    if not r :
        return f"Review is not available for this product"
    return str(r)


In [94]:
# llm call
llm = ChatGroq(model='llama-3.3-70b-versatile',temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product,get_review],
    system_prompt=" you are an helpful product assestant for an online tech store , answer in structured format  few lines under 5 lines"
)


def ask2(question:str):
    result = agent2.invoke({'messages':[{"role":'user','content':question}]})
    print(result['messages'][-1].content)

In [95]:
ask2("how do people like a smart watch")

People like a smart watch, it has 555 reviews and a rating of 4.4.


In [96]:
ask(" whats the price and review of smart watch")

The price of the smart watch is $199.99 and it has a 4.5-star rating.


## Agents with memory

In [97]:
from langgraph.checkpoint.memory import InMemorySaver

In [101]:
llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)

agent3 = create_agent(
    llm,
    tools=[get_product,get_review],
    system_prompt='you are an helpful product assistant for an online tech store.',
    checkpointer= InMemorySaver(),
)


def ask3(question : str):
    Config = {"configurable":{"thread_id":"user-alice-session-1"}}
    result = agent3.invoke(
        {'messages':[{"role":"user", "content":question}]},
        Config

    )

    print(result['messages'][-1].content)

In [102]:
ask3("what is the price of wireless headphones")

The price of the wireless headphones is $79.99. They are over-ear Bluetooth headphones with 30 hours of battery life and active noise cancellation.


In [103]:
ask3("what is the review and rating of this product")

The wireless headphones have 1234 reviews and an average rating of 4.6.
